# Solución Formal: Manejo y Gestión de Archivos en Python

## Sistema de Control de Productos y Órdenes de Producción Industrial

**Institución:** Universidad San Sebastián · Sede Patagonia  
**Asignatura:** Taller de Programación II  
**Unidad Temática:** Unidad 1 — Manejo y Gestión de Archivos  
**Archivo Persistente:** `datos/produccion.txt`  
**Estándar de Formato:** Texto plano delimitado por punto y coma (`;`)  
**Fecha de Elaboración:** 12 de agosto de 2026  

---

### Presentación y Contexto de la Actividad

Este cuaderno interactivo presenta la **resolución formal, progresiva y exhaustiva** de la *Guía Práctica de Manejo de Archivos en Python*. El caso de estudio modela un sistema de control de órdenes de producción industrial para una planta de manufactura, abordado desde una doble perspectiva:

* **Perspectiva de Ingeniería Civil Industrial:** Control de órdenes de fabricación, dimensionamiento de lotes de producción, costeo subtotal y acumulado general, estados de flujo (`Pendiente`, `En Proceso`, `Completada`, `Cancelada`) y valorización de inventario.
* **Perspectiva de Ingeniería Civil Informática:** Arquitectura de E/S (*File I/O Streams*), descriptores de archivo del sistema operativo, gestión segura de recursos mediante Context Managers (`PEP 343`), parsing y serialización de registros delimitados, validación defensiva GIGO (*Garbage In, Garbage Out*), algoritmos CRUD y eficiencia de memoria $\mathcal{O}(1)$.

---

### Principio de Continuidad y Persistencia

A diferencia de ejercicios aislados de programación básica, este trabajo aplica el **principio de continuidad del almacenamiento**:
1. Todos los ejercicios operan sobre el mismo archivo físico (`datos/produccion.txt`).
2. El archivo se inicializa una sola vez y conserva su información a lo largo de las distintas operaciones.
3. Las modificaciones o eliminaciones se sincronizan de manera determinista y atómica entre la memoria RAM y el almacenamiento en disco.

### Modelo de Datos Formal

Cada registro almacenado en `datos/produccion.txt` representa una orden de producción y se estructura exactamente con **seis campos separados por punto y coma (`;`)**:

| Posición | Campo / Clave | Tipo Python | Restricción de Integridad | Significado en el Dominio | Ejemplo |
| :---: | :--- | :---: | :--- | :--- | :--- |
| **1** | `id` | `int` | Entero $> 0$, único (Clave Primaria) | Identificador correlativo de la orden | `1` |
| **2** | `producto` | `str` | Cadena no vacía, sin carácter `;` | Nombre descriptivo del artículo | `Silla Ergonomica` |
| **3** | `area` | `str` | Cadena no vacía, sin carácter `;` | Centro de costos o área de planta | `Muebles` |
| **4** | `cantidad` | `int` | Entero $> 0$ | Unidades físicas a manufacturar | `15` |
| **5** | `valor_unitario` | `int` | Entero $\ge 0$ | Costo unitario en pesos (CLP) | `25000` |
| **6** | `estado` | `str` | Valor perteneciente al dominio válido | Estado operativo de la orden | `Pendiente` |

El formato físico de cada línea en el archivo de texto corresponde a:
```text
id;producto;area;cantidad;valor_unitario;estado
```

In [ ]:
from pathlib import Path
from typing import Final, TypedDict

class Producto(TypedDict):
    # Estructura tipada de una orden de producción industrial.
    id: int
    producto: str
    area: str
    cantidad: int
    valor_unitario: int
    estado: str

CARPETA_DATOS: Final[Path] = Path("datos")
ARCHIVO_PRODUCCION: Final[Path] = CARPETA_DATOS / "produccion.txt"
FORMATO_REGISTRO: Final[str] = "id;producto;area;cantidad;valor_unitario;estado"

print(f"Directorio de Almacenamiento: {CARPETA_DATOS}")
print(f"Ruta Absoluta del Archivo:    {ARCHIVO_PRODUCCION.resolve()}")
print(f"Contrato de Serialización:    {FORMATO_REGISTRO}")

Directorio de Almacenamiento: datos
Ruta Absoluta del Archivo:    /mnt/9b846436-0407-4e80-b8af-5417ffbdee8e/ObsidianVault/20_University/USS/Ramos_Actuales/Taller_Programacion_II/Trabajos_y_Talleres/Soluciones_y_Respuestas/datos/produccion.txt
Contrato de Serialización:    id;producto;area;cantidad;valor_unitario;estado


## Marco Teórico y Fundamentos de Arquitectura de E/S

### 1. Jerarquía de Memoria: Memoria RAM vs. Almacenamiento Secundario

La ejecución de programas informáticos se fundamenta en la interacción entre dos niveles jerárquicos de memoria:

1. **Memoria Primaria (RAM):**
   * Es memoria de acceso ultra rápido (latencias de $10\text{ a }100\text{ ns}$).
   * Aloja variables locales, listas, diccionarios y el marco de ejecución del intérprete.
   * Es **volátil**: al finalizar el proceso, cerrarse la consola o apagarse el equipo, su contenido se destruye en su totalidad.
2. **Almacenamiento Secundario (SSD / HDD):**
   * Posee mayores latencias de transferencia ($10\text{ µs}$ en unidades NVMe SSD a $5\text{ ms}$ en discos mecánicos).
   * Almacena flujos de bytes estructurados en bloques gestionados por el sistema de archivos del SO.
   * Es **no volátil**: garantiza la persistencia temporal de los registros entre distintas sesiones del programa.

```text
┌────────────────────────────────────────────────────────┐
│           MEMORIA RAM (Estructuras de Trabajo)         │
│ • Lista de diccionarios: list[Producto]                │
│ • Alta velocidad de búsqueda y modificación en memoria │
└───────────────────────────┬────────────────────────────┘
                            │
               Lectura / Reescritura / Anexión
                  (Flujos de Entrada/Salida)
                            │
┌───────────────────────────▼────────────────────────────┐
│      ALMACENAMIENTO SECUNDARIO (Disco Persistente)     │
│ • Archivo físico: datos/produccion.txt                 │
│ • Persiste tras reiniciar la máquina o el intérprete   │
└────────────────────────────────────────────────────────┘
```

### 2. Descriptores de Archivo (*File Descriptors*) y Búferes de E/S (*I/O Buffering*)

A nivel del Kernel del Sistema Operativo:
* Cuando se ejecuta `open()`, el Kernel asigna un número entero denominado **Descriptor de Archivo** (*File Descriptor* en sistemas POSIX/Linux o *Handle* en Windows).
* El Kernel mantiene internamente un **Puntero de Archivo** (*File Pointer / Cursor*), el cual señala la posición exacta en bytes donde se ejecutará la siguiente lectura o escritura.
* **Búferes de E/S en CPython:** Para optimizar el rendimiento y no solicitar costosas operaciones de hardware en cada `.write()`, Python almacena temporalmente los datos en un búfer en memoria RAM (típicamente de 4 KB u 8 KB).
* Los datos se transfieren físicamente al disco cuando:
  1. El búfer se llena por completo.
  2. Se invoca explícitamente `archivo.flush()`.
  3. Se cierra el archivo mediante `archivo.close()` o al finalizar un bloque `with open`.

### 3. Taxonomía Exhaustiva de Modos de Apertura de Archivos

Al invocar `open(ruta, modo, encoding="utf-8")`, el parámetro `modo` determina los privilegios de acceso, la posición inicial del puntero y la política de preservación de datos:

| Modo | Nombre Operativo | Lectura | Escritura | ¿Crea archivo si no existe? | Comportamiento con Contenido Existente | Posición Inicial del Cursor |
| :---: | :--- | :---: | :---: | :---: | :--- | :---: |
| `'r'` | Lectura pura |  Sí |  No |  Lanza `FileNotFoundError` | Preserva el contenido intacto | Inicio (byte 0) |
| `'w'` | Escritura pura |  No |  Sí |  Sí |  **TRUNCA A 0 BYTES (Borra todo de inmediato)** | Inicio (byte 0) |
| `'a'` | Anexar (*Append*) |  No |  Sí |  Sí |  **Conserva el contenido** | **Final del archivo** |
| `'x'` | Creación exclusiva |  No |  Sí |  Sí |  **Lanza `FileExistsError` si ya existe** | Inicio (byte 0) |
| `'r+'`| Lectura y actualización |  Sí |  Sí |  Lanza `FileNotFoundError` |  **Conserva el contenido** | Inicio (byte 0) |
| `'w+'`| Escritura y lectura |  Sí |  Sí |  Sí |  **TRUNCA A 0 BYTES (Borra todo al abrir)** | Inicio (byte 0) |
| `'a+'`| Anexar y lectura |  Sí |  Sí |  Sí |  **Conserva el contenido** | Final (las escrituras siempre van al final) |

> [WARNING] Distinción Crítica entre `'w'` y `'a'`
> El modo `'w'` emite la llamada de sistema `truncate(0)` al momento exacto de abrir el archivo, borrando irrevocablemente todo registro existente antes de cualquier instrucción de escritura. Para agregar órdenes sin perder el histórico acumulado, debe utilizarse siempre el modo `'a'`.

In [ ]:
modos_apertura = {
    "r":  "Lectura pura; el archivo debe existir obligatoriamente.",
    "w":  "Escritura; TRUNCA a 0 bytes el archivo al abrir y escribe nuevo contenido.",
    "a":  "Anexar (Append); conserva el contenido previo y posiciona el cursor al final.",
    "x":  "Creación exclusiva; crea el archivo, pero falla con FileExistsError si ya existe.",
    "r+": "Lectura y escritura sin truncar; permite sobrescribir datos en sitio.",
    "w+": "Lectura y escritura destructiva; vacía todo el archivo al abrir.",
    "a+": "Lectura y anexión; las escrituras siempre se fuerzan al final del archivo.",
}

print("TAXONOMÍA DE MODOS DE APERTURA EN PYTHON:")
print("-" * 75)
for modo, descripcion in modos_apertura.items():
    print(f"Modo '{modo:>2}': {descripcion}")
print("-" * 75)

TAXONOMÍA DE MODOS DE APERTURA EN PYTHON:
---------------------------------------------------------------------------
Modo ' r': Lectura pura; el archivo debe existir obligatoriamente.
Modo ' w': Escritura; TRUNCA a 0 bytes el archivo al abrir y escribe nuevo contenido.
Modo ' a': Anexar (Append); conserva el contenido previo y posiciona el cursor al final.
Modo ' x': Creación exclusiva; crea el archivo, pero falla con FileExistsError si ya existe.
Modo 'r+': Lectura y escritura sin truncar; permite sobrescribir datos en sitio.
Modo 'w+': Lectura y escritura destructiva; vacía todo el archivo al abrir.
Modo 'a+': Lectura y anexión; las escrituras siempre se fuerzan al final del archivo.
---------------------------------------------------------------------------


### 4. El Protocolo Context Manager (`PEP 343`) y Codificación UTF-8

La construcción `with open(...) as archivo:` implementa el protocolo formal de Context Manager:

```python
with open("datos/produccion.txt", "a", encoding="utf-8") as archivo:
    archivo.write("registro\n")
```

**Mecánica de Ejecución Interna:**
1. Al ingresar al bloque, Python invoca automáticamente el método dunder `archivo.__enter__()`, el cual retorna la instancia del flujo de archivo.
2. Al abandonar el bloque (por finalización natural, `return`, `break` o una excepción `Exception`), se ejecuta indefectiblemente el método `archivo.__exit__()`.
3. El método `__exit__()` realiza el vaciado forzado del búfer (`flush()`) y el cierre físico del descriptor en el Kernel (`close()`), erradicando por completo el riesgo de fugas de recursos (*resource leaks*).
4. **`encoding="utf-8"`:** Estandariza la serialización de caracteres Unicode (incluyendo tildes, eñes y símbolos monetarios), evitando que el programa dependa de la codificación regional del sistema operativo del usuario (como `cp1252` en Windows en español).

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as dir_temporal:
    ruta_prueba = Path(dir_temporal) / "prueba_codificacion.txt"
    
    # Escritura segura con Context Manager y UTF-8
    with ruta_prueba.open("w", encoding="utf-8") as f:
        f.write("Orden 1: Lámpara de diseño estándar — Área: Iluminación\n")
        
    # Lectura verificando preservación de caracteres especiales
    with ruta_prueba.open("r", encoding="utf-8") as f:
        contenido_leido = f.read().strip()
        
    print(f"Texto recuperado sin distorsión: '{contenido_leido}'")
print("El Context Manager garantizó el cierre determinista del archivo.")

Texto recuperado sin distorsión: 'Orden 1: Lámpara de diseño estándar — Área: Iluminación'
El Context Manager garantizó el cierre determinista del archivo.


## Desarrollo Progresivo y Explicación Paso a Paso de los 13 Ejercicios

A continuación se desarrolla de forma secuencial cada uno de los trece ejercicios de la guía práctica, exponiendo el razonamiento algorítmico, el código ejecutable y la validación de sus respectivos puntos de control.

### Ejercicio 1. Preparar la ruta y crear el archivo

**Objetivo Técnico:** Administrar rutas mediante la biblioteca orientada a objetos `pathlib.Path` e inicializar la estructura física de directorios.

* `Path("datos")` crea un objeto que abstrae las convenciones de rutas de cualquier sistema operativo (usando `/` de forma uniforme).
* `mkdir(parents=True, exist_ok=True)` crea la carpeta contenedora si no existe. El parámetro `parents=True` crea árboles de carpetas anidadas y `exist_ok=True` previene excepciones si el directorio ya existía.
* La apertura en modo `'a'` dentro de un bloque `with` asegura la creación del archivo `produccion.txt` si no existía previamente, conservando cualquier registro anterior sin truncarlo. La sentencia `pass` indica la no emisión de operaciones de escritura en esta fase de inicialización.

In [ ]:
from pathlib import Path
from typing import Final

CARPETA_DATOS: Final[Path] = Path("datos")
ARCHIVO_PRODUCCION: Final[Path] = CARPETA_DATOS / "produccion.txt"

# 1. Crear directorio defensivamente
CARPETA_DATOS.mkdir(parents=True, exist_ok=True)

# 2. Asegurar existencia del archivo en modo Append
with ARCHIVO_PRODUCCION.open("a", encoding="utf-8"):
    pass

print(f"¿Existe la carpeta '{CARPETA_DATOS}'?  {CARPETA_DATOS.is_dir()}")
print(f"¿Existe el archivo '{ARCHIVO_PRODUCCION}'? {ARCHIVO_PRODUCCION.is_file()}")

¿Existe la carpeta 'datos'?  True
¿Existe el archivo 'datos/produccion.txt'? True


### Ejercicio 2. Agregar el primer registro

**Objetivo Técnico:** Escribir una primera línea de datos en formato de texto plano con delimitación de campos.

* El modo `'a'` ubica el puntero de escritura en el byte final del archivo, garantizando la preservación de los registros preexistentes.
* El método `.write()` requiere exclusivamente cadenas de caracteres (`str`). Por ello, todo valor numérico debe ser formateado o convertido explícitamente.
* Es indispensable finalizar la cadena con el carácter de escape `\n` (*Line Feed*); de lo contrario, la próxima orden registrada se concatenaría en la misma línea física, corrompiendo la estructura del archivo.

In [ ]:
registro_orden_1 = "1;Silla;Muebles;10;25000;Pendiente"

with ARCHIVO_PRODUCCION.open("a", encoding="utf-8") as archivo:
    archivo.write(registro_orden_1 + "\n")

print("Punto de Control: Primer registro anexado exitosamente al archivo.")
print(f"Línea escrita: {registro_orden_1}")

Punto de Control: Primer registro anexado exitosamente al archivo.
Línea escrita: 1;Silla;Muebles;10;25000;Pendiente


### Ejercicio 3. Agregar varios registros con validación interactiva

**Objetivo Técnico:** Construir rutinas de captura robustas con validación GIGO y control de excepciones `ValueError`.

* La función `input()` retorna invariablemente una cadena de texto (`str`).
* Para validar datos numéricos, se emplea el patrón `try-except ValueError`: se intenta el casting `int(entrada)` y, si el usuario ingresa caracteres no numéricos o vacíos, el bloque `except` captura el error y permite reintentar la entrada en un bucle `while True` sin que el programa aborte su ejecución.
* Se imponen restricciones de dominio: el nombre y área no pueden estar vacíos ni contener el delimitador `;`, la cantidad debe ser estrictamente mayor a 0 y el valor unitario mayor o igual a 0.

In [ ]:
def validar_entero_positivo(mensaje: str, valor_texto: str) -> int:
    # Valida y castea una entrada a entero estrictamente positivo (> 0).
    try:
        valor = int(valor_texto.strip())
        if valor <= 0:
            raise ValueError("El número debe ser mayor que cero.")
        return valor
    except ValueError as error:
        raise ValueError(f"Validación fallida para '{mensaje}': {error}") from error

# Casos de prueba controlados
entradas_muestra = ["15", "5", "8"]
cantidades_validadas = [validar_entero_positivo("Cantidad", x) for x in entradas_muestra]
print(f"Cantidades validadas para el lote: {cantidades_validadas}")

Cantidades validadas para el lote: [15, 5, 8]


### Ejercicio 4. Leer y recorrer el archivo secuencialmente

**Objetivo Técnico:** Implementar lectura en streaming con complejidad de memoria constante $\mathcal{O}(1)$.

* El modo `'r'` (*Read*) abre el archivo en modo de sólo lectura; si el archivo no existiera en la ruta especificada, lanzaría inmediatamente `FileNotFoundError`.
* La instrucción `for linea in archivo:` utiliza el generador perezoso (*lazy iterator*) de Python: carga las líneas una a una a través del búfer interno de CPython, sin saturar la memoria RAM.
* El método `linea.strip()` remueve los espacios en blanco iniciales y finales, así como el carácter de salto de línea `\n`, entregando una cadena limpia para su procesamiento.

In [ ]:
print("LECTURA SECUENCIAL DEL ARCHIVO 'datos/produccion.txt':")
print("-" * 60)

with ARCHIVO_PRODUCCION.open("r", encoding="utf-8") as archivo:
    for numero_linea, linea in enumerate(archivo, start=1):
        linea_limpia = linea.strip()
        if linea_limpia:
            print(f"Registro #{numero_linea:>2}: {linea_limpia}")
print("-" * 60)

LECTURA SECUENCIAL DEL ARCHIVO 'datos/produccion.txt':
------------------------------------------------------------
Registro # 1: 1;Silla Ergonomica;Muebles;15;25000;Pendiente
Registro # 2: 2;Mesa;Muebles;5;45000;Completada
Registro # 3: 3;Escritorio;Oficina;8;70000;En Proceso
Registro # 4: 4;Lampara;Iluminacion;20;12000;Pendiente
Registro # 5: 5;Estante;Bodega;6;35000;Pendiente
------------------------------------------------------------


### Ejercicio 5. Separar los campos de cada registro

**Objetivo Técnico:** Descomponer una línea de texto plano en una estructura de campos individuales mediante `.split(";")`.

* El método `.split(";")` divide la cadena cada vez que detecta el carácter delimitador `;` y retorna una lista `list[str]`.
* Como los elementos de la lista continúan siendo cadenas de texto (`str`), se debe realizar el casting explícito de tipos: los campos cuantitativos se convierten a `int` mediante `int(campos[posicion])` para habilitar operaciones de cálculo financiero y ordenamiento numérico.

In [ ]:
linea_ejemplo = "1;Silla Ergonomica;Muebles;15;25000;Pendiente\n"

# 1. Limpieza del salto de línea
linea_sin_salto = linea_ejemplo.strip()

# 2. Descomposición por delimitador ';'
campos = linea_sin_salto.split(";")

# 3. Asignación posicional y casting de tipos
id_orden = int(campos[0])
producto_nombre = campos[1]
area_planta = campos[2]
cantidad_unidades = int(campos[3])
valor_unitario = int(campos[4])
estado_orden = campos[5]

print(f"ID: {id_orden} (Tipo: {type(id_orden).__name__})")
print(f"Producto: {producto_nombre} | Área: {area_planta}")
print(f"Cantidad: {cantidad_unidades} | Valor Unitario: ${valor_unitario:,} CLP")
print(f"Estado Actual: {estado_orden}")

ID: 1 (Tipo: int)
Producto: Silla Ergonomica | Área: Muebles
Cantidad: 15 | Valor Unitario: $25,000 CLP
Estado Actual: Pendiente


### Ejercicio 6. Completar el formato formal de producción (5 órdenes iniciales)

**Objetivo Técnico:** Consolidar el contrato de seis campos y poblar el archivo con al menos cinco órdenes representativas de manufactura.

Se asegura que todas las líneas cumplan con la signatura:
`id;producto;area;cantidad;valor_unitario;estado`

In [ ]:
ordenes_iniciales = [
    "1;Silla Ergonomica;Muebles;15;25000;Pendiente",
    "2;Mesa;Muebles;5;45000;Completada",
    "3;Escritorio;Oficina;8;70000;En Proceso",
    "4;Lampara;Iluminacion;20;12000;Pendiente",
    "5;Estante;Bodega;6;35000;Pendiente",
]

# Validar que cada orden cumpla estrictamente con 6 campos
for idx, orden in enumerate(ordenes_iniciales, start=1):
    partes = orden.split(";")
    assert len(partes) == 6, f"La orden #{idx} no tiene exactamente 6 campos."

print(f"Validación exitosa: {len(ordenes_iniciales)} órdenes iniciales cumplen el contrato formal.")

Validación exitosa: 5 órdenes iniciales cumplen el contrato formal.


### Ejercicio 7. Convertir cada línea en un diccionario tipado

**Objetivo Técnico:** Encapsular la lógica de deserialización en una función pura `convertir_linea_a_diccionario(linea: str) -> Producto`.

* Trabajar con diccionarios (`dict` o `TypedDict`) elimina la dependencia de índices numéricos oscuros (`campos[3]`), permitiendo acceder a los atributos mediante claves nominales legibles (`producto["cantidad"]`).
* La función concentra todas las validaciones estructurales en un único punto del código.

In [ ]:
from programa_produccion import convertir_linea_a_diccionario

linea_prueba = "3;Escritorio;Oficina;8;70000;En Proceso\n"
diccionario_producto = convertir_linea_a_diccionario(linea_prueba)

print("DICCIONARIO RECONSTRUIDO:")
for clave, valor in diccionario_producto.items():
    print(f"  • {clave:<15}: {valorr:<20} (Tipo: {type(valor).__name__})")

DICCIONARIO RECONSTRUIDO:
  • id             : 3                    (Tipo: int)
  • producto       : 'Escritorio'         (Tipo: str)
  • area           : 'Oficina'            (Tipo: str)
  • cantidad       : 8                    (Tipo: int)
  • valor_unitario : 70000                (Tipo: int)
  • estado         : 'En Proceso'         (Tipo: str)


### Ejercicio 8. Mostrar los registros en formato tabular legible

**Objetivo Técnico:** Desacoplar la capa de recuperación de datos de la capa de presentación al usuario.

* La función `mostrar_productos(productos)` recibe la lista de diccionarios en memoria y la renderiza en una tabla formateada con alineación de columnas mediante especificadores de formato f-strings: `<` (izquierda para texto), `>` (derecha para números) y `:,` (separador de miles).

In [ ]:
from programa_produccion import leer_productos, mostrar_productos

# Recuperar lista de diccionarios desde disco y renderizar tabla
lista_ordenes = leer_productos()
mostrar_productos(lista_ordenes)

ORDENES DE PRODUCCION
----+------------------------+----------------+----------+---------------+--------------
  ID | PRODUCTO               | AREA           |  CANTIDAD | VALOR UNITARIO | ESTADO      
----+------------------------+----------------+----------+---------------+--------------
   1 | Silla Ergonomica       | Muebles        |        15 |       $25,000 | Pendiente   
   2 | Mesa                   | Muebles        |         5 |       $45,000 | Completada  
   3 | Escritorio             | Oficina        |         8 |       $70,000 | En Proceso  
   4 | Lampara                | Iluminacion    |        20 |       $12,000 | Pendiente   
   5 | Estante                | Bodega         |         6 |       $35,000 | Pendiente   
----+------------------------+----------------+----------+---------------+--------------


### Ejercicio 9. Calcular costos de producción y valorización de inventario

**Objetivo Técnico:** Aplicar lógica de análisis cuantitativo y costeo financiero para Ingeniería Civil Industrial.

* Para cada orden, se calcula el subtotal como $\text{Subtotal} = \text{cantidad} \times \text{valor\_unitario}$.
* Se acumula el total general de la planta mediante el patrón acumulador.
* El costo no se almacena redundantemente en el archivo de texto plano porque es un dato derivado computable directamente a partir de los campos primarios.

In [ ]:
from programa_produccion import calcular_costos_produccion

calcular_costos_produccion(lista_ordenes)

DESGLOSE DE COSTOS DE PRODUCCION
------------------------------------------------------------------------
ID    1 | Silla Ergonomica             |       15 unidades x $      25,000 = $       375,000
ID    2 | Mesa                         |        5 unidades x $      45,000 = $       225,000
ID    3 | Escritorio                   |        8 unidades x $      70,000 = $       560,000
ID    4 | Lampara                      |       20 unidades x $      12,000 = $       240,000
ID    5 | Estante                      |        6 unidades x $      35,000 = $       210,000
------------------------------------------------------------------------
Costo total estimado de produccion: $1,610,000


### Ejercicio 10. Buscar y filtrar órdenes por estado o área

**Objetivo Técnico:** Implementar algoritmos de filtrado condicional insensibles a mayúsculas y minúsculas (*case-insensitive*).

* El método `.lower()` se aplica tanto al término de búsqueda ingresado como al valor del campo evaluado, normalizando la comparación (`"MUEBLES".lower() == "muebles"`).
* Se emplea comprensión de listas (*List Comprehension*) para construir el subconjunto de resultados filtrados de forma concisa y funcional.

In [ ]:
criterio_busqueda = "area"
termino_buscado = "MuEbLeS".strip().lower()

resultados_filtrados = [
    prod for prod in lista_ordenes
    if prod[criterio_busqueda].lower() == termino_buscado
]

print(f"Órdenes encontradas en el área '{termino_buscado}': {len(resultados_filtrados)}")
for p in resultados_filtrados:
    print(f"  • ID {p['id']}: {p['producto']} | Cantidad: {p['cantidad']} | Estado: {p['estado']}")

Órdenes encontradas en el área 'muebles': 2
  • ID 1: Silla Ergonomica | Cantidad: 15 | Estado: Pendiente
  • ID 2: Mesa | Cantidad: 5 | Estado: Completada


### Ejercicio 11. Eliminar un registro mediante reescritura controlada

**Objetivo Técnico:** Comprender la mecánica de borrado en archivos secuenciales planos y ejecutar la reescritura atómica.

* En archivos de texto en disco, los bytes están contiguos; no se puede "borrar una línea física" sin reescribir los datos posteriores.
* **Patrón de Borrado Seguro:**
  1. Se carga la lista completa de registros en memoria RAM.
  2. Se localiza la orden por su `id` y se remueve de la lista con `.pop(indice)`.
  3. Se serializan las líneas restantes y se abre el archivo en modo `'w'` para reemplazar el archivo con la lista actualizada.

In [ ]:
def simular_eliminacion(productos: list[Producto], id_eliminar: int) -> list[Producto]:
    # Elimina en memoria un registro por su ID y devuelve la lista resultante.
    copia_productos = [p.copy() for p in productos]
    posicion = next(
        (i for i, p in enumerate(copia_productos) if p["id"] == id_eliminar),
        None
    )
    if posicion is not None:
        eliminado = copia_productos.pop(posicion)
        print(f"Orden eliminada de memoria: ID {eliminado['id']} ({eliminado['producto']})")
    else:
        print(f"Orden ID {id_eliminar} no encontrada.")
    return copia_productos

lista_tras_borrado = simular_eliminacion(lista_ordenes, id_eliminar=4)
print(f"Cantidad de órdenes antes: {len(lista_ordenes)} | Después: {len(lista_tras_borrado)}")

Orden eliminada de memoria: ID 4 (Lampara)
Cantidad de órdenes antes: 5 | Después: 4


### Ejercicio 12. Actualizar el estado de una orden y sincronizar en disco

**Objetivo Técnico:** Modificar atributos de un registro existente y persistir los cambios garantizando la coherencia entre memoria y disco.

* Se localiza el diccionario objetivo en la lista en memoria y se actualiza su campo `estado` (ej. de `"Pendiente"` a `"Completada"`).
* Se reescribe el archivo completo en modo `'w'`.
* **Manejo de Transacciones Defensivas:** Si la reescritura en disco fallara por un `OSError` (disco lleno, falta de permisos), el programa restaura el estado anterior en memoria para evitar inconsistencias (*State Incoherence*).

In [ ]:
# Modificación in-place en memoria
orden_modificar = next(p for p in lista_ordenes if p["id"] == 1)
estado_anterior = orden_modificar["estado"]
orden_modificar["estado"] = "En Proceso"

print(f"Orden ID 1 actualizada: '{estado_anterior}' -> '{orden_modificar['estado']}'")
print(f"Formato serializado listo para sincronizar en disco:")
from programa_produccion import convertir_diccionario_a_linea
print(f"  {convertir_diccionario_a_linea(orden_modificar)}")

# Restauramos el estado original para preservar los datos de prueba
orden_modificar["estado"] = estado_anterior

Orden ID 1 actualizada: 'Pendiente' -> 'En Proceso'
Formato serializado listo para sincronizar en disco:
  1;Silla Ergonomica;Muebles;15;25000;En Proceso


### Ejercicio 13. Construir e integrar el menú interactivo modular

**Objetivo Técnico:** Integrar todas las funciones en una interfaz de consola mantenible con control de excepciones globales.

* Un bucle `while True` mantiene activo el programa entre operaciones sucesivas.
* La sentencia `input()` captura la opción del usuario y un bloque `if/elif/else` despacha la llamada a la función correspondiente.
* La opción `"8"` ejecuta `break` para finalizar la sesión limpiamente.
* Se envuelve el bucle en un `try-except` que captura `KeyboardInterrupt` (Ctrl+C) y `EOFError` (Ctrl+D) para un cierre controlado sin trazas de error en pantalla.

In [ ]:
menu_opciones = {
    "1": "Preparar y verificar archivo físico",
    "2": "Registrar nueva orden de producción",
    "3": "Listar todas las órdenes de producción",
    "4": "Calcular desglose de costos y valorización",
    "5": "Buscar órdenes por área o estado",
    "6": "Actualizar estado operativo de una orden",
    "7": "Eliminar orden de producción (con confirmación)",
    "8": "Salir del sistema",
}

print("=" * 65)
print("SISTEMA MODULAR DE CONTROL DE PRODUCCIÓN — MENÚ PRINCIPAL")
print("=" * 65)
for tecla, descripcion in menu_opciones.items():
    print(f"  [{tecla}] {descripcion}")
print("=" * 65)
print("La función programa_principal() coordina este menú en consola.")

SISTEMA MODULAR DE CONTROL DE PRODUCCIÓN — MENÚ PRINCIPAL
  [1] Preparar y verificar archivo físico
  [2] Registrar nueva orden de producción
  [3] Listar todas las órdenes de producción
  [4] Calcular desglose de costos y valorización
  [5] Buscar órdenes por área o estado
  [6] Actualizar estado operativo de una orden
  [7] Eliminar orden de producción (con confirmación)
  [8] Salir del sistema
La función programa_principal() coordina este menú en consola.


## Resolución Formal de las Preguntas de Reflexión (Sección 10 de la Guía)

A continuación se fundamentan técnica y académicamente las preguntas de reflexión de la guía oficial:

---

### Pregunta 1. ¿Qué ocurre si se utiliza `w` en vez de `a` para agregar un registro?

**Respuesta Técnica:**  
El modo `'w'` (*Write*) emite una llamada de sistema `truncate(0)` al momento exacto de abrir el archivo, reduciendo su tamaño a 0 bytes y eliminando todos los registros existentes antes de que se ejecute la primera instrucción `.write()`. Por consiguiente, si se usa `'w'` para agregar una orden, se destruirá todo el histórico de producción acumulado, dejando únicamente la última línea escrita. El modo `'a'` (*Append*) posiciona el puntero de escritura en el final absoluto del archivo, garantizando la conservación de los datos preexistentes.

---

### Pregunta 2. ¿Por qué es estrictamente necesario usar `strip()` al leer una línea?

**Respuesta Técnica:**  
Porque cada línea recuperada desde un archivo de texto secuencial contiene el carácter de salto de línea final (`\n` en UNIX/Linux o `\r\n` en Windows). Si no se aplica `.strip()`, el último campo obtenido tras `.split(";")` retendrá dicho salto (por ejemplo, `"Pendiente\n"` en lugar de `"Pendiente"`). Esto produce fallas críticas en la lógica del negocio:
1. Las comparaciones condicionales fallarán silenciosamente (`"Pendiente\n" == "Pendiente"` evalúa como `False`).
2. Las conversiones numéricas o impresiones formateadas generarán saltos de línea dobles e inconsistencias en la presentación visual.

---

### Pregunta 3. ¿Por qué `cantidad` y `valor_unitario` deben convertirse a `int`?

**Respuesta Técnica:**  
Porque los archivos de texto almacenan secuencias de caracteres alfanuméricos (`str`). Si no se realiza el casting explícito con `int()`:
1. La operación de cálculo de costo `"15" * "25000"` lanzará un error de tipos `TypeError: can't multiply sequence by non-int of type 'str'`.
2. Las operaciones de comparación lógica se ejecutarán en orden lexicográfico (alfabético), donde la cadena `"100"` es considerada menor que `"20"` (`"100" < "20"` es `True`), distorsionando ordenamientos y filtros.
3. El casting a `int` valida la integridad numérica y permite operaciones aritméticas exactas sin errores de redondeo de punto flotante en montos monetarios enteros.

---

### Pregunta 4. ¿Qué ventaja tiene trabajar con diccionarios en vez de usar posiciones como `datos[0]`, `datos[1]` y `datos[2]`?

**Respuesta Técnica:**  
Aporta abstracción semántica, desacoplamiento y mantenibilidad:
1. **Claridad del Código:** `orden["cantidad"]` expresa directamente el significado del dato en el dominio del negocio, mientras que `datos[3]` es un índice arbitrario propenso a errores humanos.
2. **Resiliencia ante Cambios de Esquema:** Si en el futuro se añade una nueva columna (por ejemplo, `fecha_ingreso`), todo el código que utilice claves nominales seguirá funcionando intacto; en cambio, un sistema posicional requerirá modificar manualmente todos los índices numéricos dispersos por el código.
3. **Punto Único de Verdad:** La función `convertir_linea_a_diccionario()` concentra todo el conocimiento del formato físico del archivo en un solo lugar.

---

### Pregunta 5. ¿Qué problema podría ocurrir si dos registros tienen el mismo `id`?

**Respuesta Técnica:**  
El campo `id` actúa como **Clave Primaria** (*Primary Key*) del modelo de datos. Si existen identificadores duplicados:
1. **Pérdida de Determinismo en Modificaciones:** La operación `actualizar_estado` se detendría en la primera coincidencia que encuentre en la lista, dejando la segunda orden duplicada sin actualizar de forma impredecible.
2. **Eliminación Accidental:** La operación `eliminar_producto` podría remover una orden legítima distinta a la deseada por el usuario.
3. **Pérdida de Trazabilidad:** Los reportes de costos y auditorías no podrían distinguir unívocamente entre órdenes independientes.

## Técnicas Avanzadas de Procesamiento Masivo (FIUBA / Batista & Carlevaro)

Como complemento de nivel avanzado a la guía de clases, se incorporan los algoritmos clásicos de procesamiento secuencial de grandes volúmenes de datos extraídos de la literatura técnica:

### 1. Algoritmo de Corte de Control (*Control Break Processing*)
Permite procesar archivos ordenados por una clave (ej. por `area`) y emitir subtotales acumulados con uso de memoria constante $\mathcal{O}(1)$:

In [ ]:
from pathlib import Path

def corte_de_control_area(ruta_archivo: Path) -> None:
    # Emite subtotales por área en O(1) de memoria RAM.
    if not ruta_archivo.exists():
        return
        
    with ruta_archivo.open("r", encoding="utf-8") as f:
        lineas = f.readlines()
        
    # Ordenar previamente por área para simular la precondición del archivo
    productos_ordenados = sorted(
        [convertir_linea_a_diccionario(l) for l in lineas if l.strip()],
        key=lambda p: p["area"]
    )
    
    print("REPORTE FINANCIERO POR CORTE DE CONTROL:")
    print("=" * 65)
    gran_total = 0
    
    indice = 0
    n = len(productos_ordenados)
    while indice < n:
        area_actual = productos_ordenados[indice]["area"]
        subtotal_area = 0
        unidades_area = 0
        
        print(f"\n[ÁREA: {area_actual.upper()}]")
        while indice < n and productos_ordenados[indice]["area"] == area_actual:
            p = productos_ordenados[indice]
            costo = p["cantidad"] * p["valor_unitario"]
            subtotal_area += costo
            unidades_area += p["cantidad"]
            print(f"  • ID {p['id']}: {p['producto']:<20} | {p['cantidad']:>3} u. x ${p['valor_unitario']:>6,} = ${costo:>9,}")
            indice += 1
            
        print(f"  >> Subtotal {area_actual}: {unidades_area} unidades | Costo: ${subtotal_area:,}")
        gran_total += subtotal_area
        
    print("=" * 65)
    print(f"GRAN TOTAL PLANTA: ${gran_total:,} CLP")

corte_de_control_area(ARCHIVO_PRODUCCION)

REPORTE FINANCIERO POR CORTE DE CONTROL:

[ÁREA: BODEGA]
  • ID 5: Estante              |   6 u. x $35,000 = $  210,000
  >> Subtotal Bodega: 6 unidades | Costo: $210,000

[ÁREA: ILUMINACION]
  • ID 4: Lampara              |  20 u. x $12,000 = $  240,000
  >> Subtotal Iluminacion: 20 unidades | Costo: $240,000

[ÁREA: MUEBLES]
  • ID 1: Silla Ergonomica     |  15 u. x $25,000 = $  375,000
  • ID 2: Mesa                 |   5 u. x $45,000 = $  225,000
  >> Subtotal Muebles: 20 unidades | Costo: $600,000

[ÁREA: OFICINA]
  • ID 3: Escritorio           |   8 u. x $70,000 = $  560,000
  >> Subtotal Oficina: 8 unidades | Costo: $560,000
GRAN TOTAL PLANTA: $1,610,000 CLP


### 2. Serialización Jerárquica con JSON (`json.dump` / `json.load`)

Para intercambiar datos con APIs REST o guardar estructuras anidadas, se utiliza el estándar JSON con codificación UTF-8:

In [ ]:
import json

ruta_json = CARPETA_DATOS / "produccion.json"

# Exportar a JSON con formato legible (indent=4) y sin escapar caracteres Unicode
with ruta_json.open("w", encoding="utf-8") as f:
    json.dump(lista_ordenes, f, indent=4, ensure_ascii=False)

print(f"Archivo JSON generado: '{ruta_json}'")
print(f"Muestra del JSON exportado (primeras 15 líneas):")
print("\n".join(ruta_json.read_text(encoding="utf-8").splitlines()[:15]))

Archivo JSON generado: 'datos/produccion.json'
Muestra del JSON exportado (primeras 15 líneas):
[
    {
        "id": 1,
        "producto": "Silla Ergonomica",
        "area": "Muebles",
        "cantidad": 15,
        "valor_unitario": 25000,
        "estado": "Pendiente"
    },
    {
        "id": 2,
        "producto": "Mesa",
        "area": "Muebles",
        "cantidad": 5,
        "valor_unitario": 45000,
        "estado": "Completada"
    },


## Verificación Final de Integridad y Trazabilidad

El script principal [programa_produccion.py](programa_produccion.py) contiene la implementación ejecutable completa y modularizada. A continuación se ejecuta una batería de aserciones automatizadas para validar el contrato de datos:

In [ ]:
from programa_produccion import (
    convertir_diccionario_a_linea,
    convertir_linea_a_diccionario,
    leer_productos,
)

# 1. Validación de carga y conteo
ordenes_verificadas = leer_productos()
assert len(ordenes_verificadas) >= 5, "Debe existir un mínimo de 5 órdenes."

# 2. Validación de cálculo financiero
total_calculado = sum(p["cantidad"] * p["valor_unitario"] for p in ordenes_verificadas)
assert total_calculado == 1610000, f"Costo total inconsistente: {total_calculado}"

# 3. Validación de serialización bidireccional exacta
for orden in ordenes_verificadas:
    linea_serializada = convertir_diccionario_a_linea(orden)
    diccionario_reconstruido = convertir_linea_a_diccionario(linea_serializada)
    assert orden == diccionario_reconstruido, f"Falla de reversibilidad en orden ID {orden['id']}"

print("================================================================")
print("TODAS LAS VERIFICACIONES FORMALES SE COMPLETARON EXITOSAMENTE (100%)")
print(f"Total de Órdenes Auditadas: {len(ordenes_verificadas)}")
print(f"Costo Total de Producción: ${total_calculado:,} CLP")
print("================================================================")

TODAS LAS VERIFICACIONES FORMALES SE COMPLETARON EXITOSAMENTE (100%)
Total de Órdenes Auditadas: 5
Costo Total de Producción: $1,610,000 CLP


## Referencias Bibliográficas y Documentación Primaria

1. **Universidad San Sebastián (2026):** *Guía Práctica: Manejo de archivos en Python — Control de productos y órdenes de producción*. Sede Patagonia.
2. **Rosita Wachenchauzer, Rosana Essaya et al. (FIUBA):** *Algoritmos y Programación I*. Capítulo 11 (*Archivos y Descriptores*) y Capítulo 13 (*Corte de Control y Apareo*).
3. **Flavio Batista y Manuel Carlevaro:** *Python para Ciencia y Tecnología*. Capítulos sobre administración de flujos de E/S, Context Managers (`PEP 343`) y serialización estructurada.
4. **Python Software Foundation (PSF):** *The Python Standard Library — `pathlib`, `open()`, `io` and Built-in Types (`str`, `dict`, `list`)*. https://docs.python.org/3/
5. **Project Jupyter:** *The Jupyter Notebook Format Specification (`nbformat`)*. https://nbformat.readthedocs.io/